# Advisor review package -- opx ML thermobarometer

**For:** Dr. Kanani K.M. Lee
**From:** Ta Quang Nhan (cadet, USCGA)
**Date:** 2026-04-20
**Target venue:** JGR ML & Computation

## What this document is

A single reviewable snapshot of the pyroxene thermobarometer project.
Figures come first so you can scan the results visually; data tables
come after in case you want to check a specific number.

## The project in one paragraph (plain English)

We are trying to estimate the pressure (P) and temperature (T) at
which an orthopyroxene (opx) crystal grew, using only the chemistry
of that crystal (and optionally its surrounding liquid). Classical
petrologic thermobarometers like Putirka (2008) fit a small
polynomial to experimental data; we fit a flexible machine-learning
model to the same experimental data and compare. The question we
are asking is: *does the ML model do better than the polynomial,
and if so where and by how much?*

## A quick glossary (once, then we move on)

- **opx / cpx**: orthopyroxene and clinopyroxene, two common
  pyroxene minerals.
- **opx-liq / opx-only**: two prediction pipelines. "opx-liq"
  uses both the crystal and the surrounding liquid as input;
  "opx-only" uses just the crystal chemistry.
- **pressure regime**: we pre-registered four geologic pressure
  bins before any modeling: shallow-crustal (<5 kbar),
  deep-crustal / MASH (5-15 kbar), lithospheric-mantle (15-30
  kbar), deeper-mantle (30+ kbar), plus "ALL" for the combined
  test set.
- **RMSE**: root-mean-squared error. How far, on average, a
  prediction misses the truth. Lower is better. Units: C for
  temperature, kbar for pressure.
- **bias correction**: a small post-processing step applied after
  the ML model makes its raw prediction. Two flavors are tested:
  "Form A" learns a regression of residual on predicted value
  separately in each pressure regime; "Form B" is a single smooth
  piecewise-sigmoid on the predicted-value axis.
- **ship-if-better rule**: a correction only ships if it improves
  overall RMSE AND does not make any pre-registered regime
  worse. A pre-registered promise to not cherry-pick.
- **TabPFN**: a foundation model for small tabular data. In-
  context transformer architecture, no training, no tuning, no
  SHAP. We added it as a 9th baseline family.
- **Putirka (2008)**: the reference classical thermobarometer.

## How to read this document

1. **Figures first** (Section 1): 13 core figures + 1 SHAP figure.
   Each figure has a long caption explaining what it shows and
   what to look for.
2. **Supporting figures** (Section 2): 14 SI figures for the
   curious.
3. **Data tables** (Section 3): the numbers behind the figures,
   in case you want to double-check.
4. **Methods, limitations, caveats** (Sections 4-6): short
   plain-English sections.
5. **Provenance** (Section 7): git state + file inventory.

## Sections

1. Core figures (14 PDFs: 13 main + Core_12 SHAP)
2. Supporting information figures
3. Data tables
   3.1 Headline (4 opx cells)
   3.2 Eight-cell winner table (tuned + TabPFN)
   3.3 TabPFN bias-correction scoreboard
   3.4 opx-only P per-regime breakdown
   3.5 TabPFN head-to-head (pre vs post vs tuned vs Putirka)
4. Methods summary
5. Limitations
6. Caveats and reconstruction provenance
7. Provenance (git SHA, CSV inventory)
8. Pre-registration (verbatim)

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path('.').resolve().parent.parent  # deliverables/lee_package_20260420/ -> project root
RESULTS = ROOT / 'results'
FIGS = Path('./figures')

def load_csv(relpath: str) -> pd.DataFrame:
    p = ROOT / relpath
    if not p.exists():
        raise FileNotFoundError(f'required CSV missing: {p}')
    return pd.read_csv(p)

def load_json(relpath: str) -> dict:
    p = ROOT / relpath
    if not p.exists():
        raise FileNotFoundError(f'required JSON missing: {p}')
    return json.loads(p.read_text())

pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print('Setup complete; loader ready.')

## 1. Core figures

Fourteen figures in total. Each one is paired with a long caption
written in plain language. You don't need to load any data to read
this section.

- **Core_01 / Core_01b**: who is in the training set, who is in
  the held-out test set (the sanity-check that no single citation
  leaks across the split).
- **Core_02**: the citation-grouped fold split used for cross-
  validation.
- **Core_03**: a one-page flowchart of the ML pipeline.
- **Core_04**: cross-pipeline heatmap of model RMSE.
- **Core_05**: per-regime RMSE before and after bias correction.
- **Core_06**: residual-vs-prediction scatter at canonical seed 42.
- **Core_07**: scoreboard of ML vs Putirka, regime by regime.
- **Core_08**: the headline per-regime RMSE bar chart for all four
  opx cells.
- **Core_09a / Core_09b**: how each ML family compares, by regime
  and overall.
- **Core_10**: best ML model vs Putirka, one dot per test sample.
- **Core_11**: natural-sample two-pyroxene 1:1 agreement plot.
- **Core_12 (new)**: SHAP feature importance for the best-
  explainable model per cell. SHAP tells you which oxide features
  the model was actually leaning on.

### Core_01_fig_dataset_map

Figure 1. Dataset map. P-T distribution of all experiments in the ExPetDB 2025-07-21 export, partitioned into four pyroxene tracks (opx-liq, opx-only, cpx-liq, cpx-only). Points are colored by pre-registered pressure regime (shallow_crustal <5 kbar, deep_crustal_MASH 5-15 kbar, lithospheric_mantle 15-30 kbar, deeper_mantle >=30 kbar); regime edges were locked on 2026-04-17 before any correction fitting. Dashed horizontal lines mark regime boundaries (labeled at right axis). Marker opacity distinguishes the 80/20 citation-grouped train/test split: train points faded, test points with dark edges. Marginal histograms show the univariate T and P coverage. Each panel is titled with its track name. Directly below each scatter, a per-panel two-column summary reports n_total, n_train, n_test, citation count, and 1st-99th percentile T and P ranges on the left, with per-regime counts on the right. The regime/split color key is shown as a single shared legend at the bottom.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_01_fig_dataset_map.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_01b_fig_dataset_map_holdout

Figure 1b. Distribution comparison between the ExPetDB training corpus and the ArcPL external held-out dataset. Top row: ExPetDB opx_liq (panel a) and cpx_liq (panel b) P-T scatter, colored by pre-registered pressure regime with marginal T and P histograms. Bottom row: ArcPL cpx_liq held-out (panel c, regime colored) and ArcPL overlaid on the ExPetDB cpx_liq hull (panel d, gray shows ExPetDB extent, vermillion shows ArcPL). ArcPL is cpx- only (n=314 filtered to finite P/T) and serves as an out-of- distribution check for the cpx pipeline. Regime boundary dashed lines are labeled at the right edge of each plot using axis-fraction coordinates so labels always stay inside the axes. Source: data/processed/{opx_clean_opx_liq, cpx_clean_cpx_liq}.parquet and data/external/agreda_lopez_2024/.../ArcPl_filtered.xlsx.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_01b_fig_dataset_map_holdout.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_02_fig_citation_split

Figure 2. Citation-grouped cross-validation schematic on the ExPetDB training corpus (opx n=1635, cpx n=5282 after equilibrium and cation filters). (a) A naive row-random 80/20 split lets experiments from the same publication appear on both sides of the split; correlated features (same laboratory, same protocol, same starting materials) leak from train to test and inflate generalisation estimates. (b) The citation-grouped split used throughout this study: every publication is held wholly on one side. We use scikit-learn StratifiedGroupKFold with 10 folds and min_train_fold=50 to stratify on pressure regime while respecting citation grouping. Each cell represents one experiment; each row is one publication (~93 publications in the opx corpus, ~260 in the cpx corpus).

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_02_fig_citation_split.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_03_fig_methods_flowchart

Figure 3. Methods flowchart. Three phase panels, each containing numbered stages that flow strictly left-to-right. Phase I (Data): raw ExPetDB (stage 1), filtering (stage 2), citation-grouped 10-fold split using StratifiedGroupKFold with min_train_fold=50 (stage 3). Phase II (Model training): 9 model families (stage 4) split into two tuning routes at stage 5 (Optuna 200 trials, seed 42 for the eight tuned families, vs. TabPFN default at n_estimators=8), followed by parameter freeze (stage 6) and a 20-seed refit (5-seed for TabPFN to bound CPU cost) producing out-of-fold predictions (stage 7). Phase III (Evaluation): OOF bias fit for Form A (per-regime OLS) and Form B (piecewise Agreda-Lopez sigmoid) (stage 8); conservative ship-if-better rule with overall_delta > 1e-6 AND max_regime_degradation <= 1e-6 (stage 9); evaluation on two slices at stage 10 -- the held-out ExPetDB test partition split into the four pre-registered pressure regimes, plus the ArcPL n=197 external-validation dataset as an out-of-distribution check.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_03_fig_methods_flowchart.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_04_fig_nb04_cross_pipeline_heatmap

Figure 4. Overall test-set RMSE heatmap across the four opx track/target combinations (rows) versus ten candidates (columns: the nine ML model families in canonical MODEL_ORDER plus the best Putirka / Agreda external benchmark). Each cell reports the mean RMSE of the 20-seed refit (5 seeds for TabPFN) over the held-out test partition, together with the half-width of the 95% confidence interval (1.96 * std for the ML families, half the bootstrap CI for Putirka). Cells are colored via a row-normalised viridis_r colormap so the row minimum (best) is lightest and the row maximum (worst) is darkest; absolute values are printed in each cell in native units (C for T targets, kbar for P targets). Opx-only T has no Putirka opx-only thermometer available in Thermobar and is marked N/A. Per-family feature_set choice (raw / alr / pwlr) is the 20-seed best per row. The minimum-RMSE cell in each row is outlined in thin red to highlight the best model per opx combo.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_04_fig_nb04_cross_pipeline_heatmap.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_05_fig30_bias_correction_per_regime_rmse

Figure 5. Bias-correction effect on per-regime test-set RMSE for the four opx track/target combinations. Each regime group shows three bars: pre-correction (gray), post Form A (regime-piecewise OLS, blue), post Form B (Agreda-Lopez piecewise sigmoid, vermillion). Bar heights are 20-seed means; whiskers are the 20-seed mean of per-seed bootstrap 95% CIs. Hatched bars mark regimes with n < 20 (sample-size limited, interpret with care). The shipped correction form per panel (accepts Form A / accepts Form B / ships none) is annotated in the top-right; a form only ships when it improves overall RMSE and does not degrade any regime by more than 1e-6. How the math works. Both forms post-process the raw ML prediction y_hat using the out-of-fold residual e = y_hat - y learned on the training partition, then subtract the learned residual at test time (y_corr = y_hat - e_hat). Form A (regime-piecewise OLS) fits a separate linear regression e = a + b * y_hat on each pre-registered P-regime, so the correction is a constant slope/intercept within each regime (four regressions stitched end-to-end by regime boundary). Form B (Agreda-Lopez sigmoid) fits one smooth piecewise-sigmoid curve e(y_hat) across all regimes with four learned breakpoints (following Agreda-Lopez et al., 2024), which avoids the discontinuities at regime edges that Form A introduces but adds more degrees of freedom. A form "ships" only if it reduces overall RMSE and does not hurt any regime. Cpx pipelines are intentionally excluded; this paper centers the opx ML thermobarometer. Source CSVs: results/bias_correction_per_seed.csv, results/bias_correction_shipped.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_05_fig30_bias_correction_per_regime_rmse.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_06_fig31_bias_correction_residuals

Figure 6. Residual-vs-predicted scatter at canonical seed 42 for the four opx track/target combinations. Gray markers: pre-correction residuals. Colored markers: residuals after the shipped form of bias correction, colored by the sample's pre-registered P regime (shallow_crustal blue, deep_crustal_MASH orange, lithospheric_mantle green, deeper_mantle vermillion). A gray band marks +/-1 sigma of the post-correction residuals. Panels whose shipping decision is "none" show the pre-correction residuals alone and are labelled accordingly. The figure-level legend describes marker colors and applies to every panel, including the "ships none" panels where the colored markers would otherwise not appear. Source: results/bias_correction_shipped.csv and per-sample predictions from results/bias_correction/checkpoints/ (seed 42).

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_06_fig31_bias_correction_residuals.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_07_fig34_bias_correction_scorecard_delta

Figure 7. Per-regime scorecard delta between our post-correction-bias-correction RMSE and the best external Putirka (2008) thermobarometer, for the four opx track/target cells. Color encodes fractional improvement (Putirka_RMSE - v10_post_RMSE) / Putirka_RMSE on a diverging red-white-green scale: green cells are wins for the tuned ML baseline, red cells are wins for Putirka. Cell text shows the absolute RMSE delta in native units (C for T, kbar for P). External references are Putirka (2008) thermobarometers run through Thermobar: opx-liq T = eq 28a, opx-liq P = eq 29a (29b in deeper_mantle), opx-only P = eq 29c. Opx-only T has no Putirka opx-only thermometer in Thermobar, so its row reports the tuned ML baseline absolute RMSE instead of a delta. Cells marked with "*" had their post-correction RMSE flip under the v2 tiered ship rule (Amendment 1, 2026-04-20) because a low-n regime that had vetoed the correction under v1 is non-vetoing under v2. Cpx pipelines are intentionally excluded. Source: results/preregistered_scorecard_postcorrection_v2.csv (with v1 reference from preregistered_scorecard_postcorrection.csv).

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_07_fig34_bias_correction_scorecard_delta.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_08_fig45_opx_headline

Figure 8. Per-regime RMSE for all four opx track/target combinations across five candidates: pre-correction tuned baseline-correction (gray), post-correction tuned baseline-correction in its shipped form (blue), best external Putirka (2008) thermobarometer (red), TabPFN v2 pre-correction (pink), TabPFN v2 post-correction in its shipped form (green, hatched). Regimes follow the pre-registered pressure partition (<5, 5-15, 15-30, >=30 kbar) plus ALL. Error bars are 95% bootstrap CIs (n_boot = 500). "ALL winner" in each panel is the preregistered scorecard verdict, enriched with the winning model family / feature-set (e.g. ERT/pwlr = ERT trained on pairwise-log-ratio transformed oxide features). Opx-only T (panel c) has no Putirka opx-only thermometer in Thermobar, so the red bar is absent. Source: results/preregistered_scorecard_postcorrection.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_08_fig45_opx_headline.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_09a_fig_opx_regime_families

Figure 9a. Per-regime test-set RMSE for all nine model families versus the best Putirka / Agreda external benchmark, across the four opx combinations. Each regime group contains ten bars: nine colored bars (one per family, in the canonical MODEL_ORDER) plus a gray Putirka / Agreda bar. For each (family, regime) cell the feature_set (raw / alr / pwlr) with the lowest test RMSE is selected; whiskers are 95% bootstrap CIs. Pressure regimes are the pre-registered edges shallow_crustal <5 kbar, deep_crustal_MASH 5-15 kbar, lithospheric_mantle 15-30 kbar, deeper_mantle >=30 kbar, plus the ALL aggregate column. Panel (c) opx-only T has no Putirka opx-only thermometer in Thermobar and is annotated accordingly. Core_09b is the overall-only companion; Core_10 collapses to best-of-ours vs Putirka.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_09a_fig_opx_regime_families.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_09b_fig_opx_overall_families

Figure 9b. Overall test-set RMSE per model family for all four opx combinations, aggregated across the full held-out partition (no regime split). Bars show the mean 20-seed RMSE (5-seed for TabPFN), whiskers are 95% intervals computed as mean +/- 1.96 * std over seeds; the feature-set winner (raw, alr, or pwlr) for each family is printed below the bar in the family color. The dashed horizontal line is the best Putirka / Agreda external benchmark from the ALL-row of the pre-registered scorecard (shaded band = 95% bootstrap CI on the benchmark); panel (c) opx-only T has no Putirka opx-only thermometer in Thermobar and is annotated accordingly. This figure answers "which family is best overall?"; the companion Core_09a answers the same question stratified by pressure regime.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_09b_fig_opx_overall_families.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_10_fig_best_vs_putirka

Figure 10. Best of our nine model families versus Putirka / Agreda external benchmarks, per pre-registered pressure regime and aggregate ALL column, for all four opx combinations. For each (track, target, regime) the "ours" bar is the minimum pre-correction RMSE across ElasticNet, RF, ERT, GB, XGB, LightGBM, CatBoost, MLP, and TabPFN; the family whose model won that regime is labelled below its bar together with the input transform ("raw" = native oxide wt%, "alr" = additive log-ratio, "pwlr" = pairwise log-ratio) that won for that family / regime (example: ERT + pwlr). The "Putirka" bar is the best external Thermobar/Agreda benchmark from the head-to-head comparison. Whiskers are 95% bootstrap confidence intervals from the 20-seed refit (5 seeds for TabPFN). Panel (c) opx-only T has no Putirka opx-only thermometer available in Thermobar and is annotated accordingly. This is a simplified companion to Core_09 (all 9 families shown individually); Core_10 answers the narrower question "does our best ML candidate beat the best published calibration at each regime?" at a glance.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_10_fig_best_vs_putirka.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_11_fig_nb08_twopx_1to1

Figure 11. Natural paired-pyroxene sanity check on LEPR samples that contain both opx and cpx (inner-join on Experiment). Five row categories, each presented as (P 1:1, T 1:1, disequilibrium map). Row A compares our ML opx-liq (RF pwlr T, RF alr P) with our ML cpx-liq (ERT pwlr T, LightGBM pwlr P) -- cross-mineral agreement between two independently trained ML models. Row B compares our ML opx-liq with the external Agreda-Lopez (2024) cpx-liq model. Row C compares our ML opx predictions with the classical Putirka (2008) opx thermobarometers (opx-only P via eq 29c, opx-liq T via eq 28a; same-mineral benchmark). Row D compares our ML opx-liq with the Jorgenson (2022) ET-based cpx-liq model. Row E compares our ML opx-liq with the Wang (2021) XGB-based cpx-liq model. Disequilibrium maps plot DeltaP vs DeltaT; the shaded box spans +/- 2 sigma_conformal (from nb07 calibration, used as a petrological equilibrium flag). Axes are auto-scaled so no samples are clipped. Source data: LEPR_Wet_Stitched (April 2023 dump); intermediate predictions cached in results/core11_extended_predictions.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_11_fig_nb08_twopx_1to1.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### Core_12_fig_shap_winners

Figure 12. SHAP feature importance for the best-explainable tuned-family model in each of the eight pipeline x track x target cells. Each panel shows the top 10 features ranked by mean |SHAP| on the held-out test partition at canonical seed 42. Bar length is mean |SHAP| in the target's native units (C for temperature, kbar for pressure). Tuned-family identities: opx_liq T = ElasticNet/raw, opx_liq P = MLP/raw, opx_only T = LightGBM/alr, opx_only P = RF/pwlr, cpx_liq T = ERT/pwlr, cpx_liq P = LightGBM/pwlr, cpx_only T = ERT/pwlr, cpx_only P = MLP/alr. IMPORTANT: for opx_only T (panel c) and opx_only P (panel d) the 5-way scorecard (results/preregistered_scorecard_postcorrection.csv, ALL regime) promotes TabPFN post-correction over the tuned-family winner (TabPFN post-RMSE 125.6 C vs LightGBM/alr 148.0 C, and 5.94 kbar vs RF/pwlr 6.03 kbar respectively). TabPFN is an in-context foundation model and exposes no SHAP pathway (no TreeExplainer, no LinearExplainer, no gradient surface compatible with KernelExplainer at our runtime budget), so those two panels show the tuned-family runner-up's SHAP for explainability. Tree-family (RF, ERT, LightGBM) uses shap.TreeExplainer (exact); ElasticNet uses shap.LinearExplainer (exact) on scaler-transformed features; MLP uses sklearn.inspection.permutation_importance (20 repeats, seed=42) as a fast SHAP surrogate. KernelExplainer for MLP would add ~30 min per cell and was deferred in this pass (Phase 6 one-shot scope, 2026-04-20). Absolute |SHAP| values are NOT comparable across cells with different feature_sets (raw, alr, pwlr) because those transforms rescale the input. Source: results/shap_importance_winners.csv; scorecard winners from results/preregistered_scorecard_postcorrection.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'Core_12_fig_shap_winners.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

## 2. Supporting information figures

Extra figures for the curious reader. Safe to skim.

### fig24_per_regime_rmse_opx_liq

Figure 24. Per-regime RMSE (bootstrap 95% CI) for the opx-liq test split: best tuned model cell vs Putirka 2008 best equation, in the pre-registered four-bin P partition. Shaded bins are sample-size-limited (n<20).

In [ ]:
from IPython.display import Image
png = FIGS / 'fig24_per_regime_rmse_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig25_per_regime_residual_violins_opx_liq

Figure 25. Per-regime residual distributions (predicted minus observed) for the tuned ML baseline canonical opx-liq base models on the test split (T: ElasticNet/raw, P: MLP/raw), in the pre-registered P bin partition. Shaded bins are sample-size-limited (n<20); medians shown as horizontal lines.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig25_per_regime_residual_violins_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig26_generalization_opx_liq

Fig. 26. the tuned ML baseline out-of-fold generalization diagnostics for the three canonical
opx-liq cells (ElasticNet/raw/T_C aggregate winner; MLP/raw/P_kbar
aggregate winner; ElasticNet/raw/P_kbar shallow-crustal robust winner).
Each panel plots pooled held-out RMSE (95% bootstrap CI whiskers) across
four out-of-sample strategies: LeaveOneStudyOut (LOSO, Citation-grouped,
93 folds), Cluster-KFold (chemical cluster groups from NB02 k-means),
TargetBinKFold (pre-registered P-regime bins [0,5,15,30,100] kbar), and
LeaveOneRegionOut (petrologic study-type inferred from Citation text:
MORB / mantle_melting / arc_silicic / basalt / primitive_mafic /
partitioning / metamorphic / other; min_train_fold=50). Produced by
scripts/v10_phase_g_nb05_generalization.py. n_test pooled across folds
after min-train-size filtering. Supports T02/T06 (generalization claims).
Source CSVs: results/opx_liq_generalization.csv and
results/opx_liq_generalization_predictions.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig26_generalization_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig27_shap_summary_opx_liq

Fig. 27. Top-15 mean absolute SHAP values per feature for five canonical
opx-liq cells, grouped by explainer family. Linear cells use
shap.LinearExplainer (exact for ElasticNet, pipeline scaler pre-applied
to background and evaluation matrices). Tree cells (CatBoost/raw/P_kbar,
XGB/alr/T_C) use shap.TreeExplainer. The MLP/raw/P_kbar cell uses
shap.KernelExplainer with a k=50 KMeans background and an 80-sample
random subset of the test set (seed = SEED_BOOTSTRAP, nsamples=100 per
evaluation); the reported mean|SHAP| averages over the sampled rows only.
Produced by scripts/v10_phase_g_nb06_shap.py. Supports T08 (feature
attribution). Source CSV: results/opx_liq_shap_importance.csv;
per-cell SHAP arrays: results/opx_liq_shap_values.npz.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig27_shap_summary_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig28_bias_correction_opx_liq

Fig. 28. Per-regime piecewise linear bias correction for the opx-liq
canonical cells. Correction coefficients (y_corrected = a * y_pred + b)
are fit on train-set out-of-fold predictions (StratifiedGroupKFold 10
folds, Citation-grouped, target stratified by NB03 bins) and applied to
the held-out test set per pre-registered P regime (shallow_crustal
[0,5) kbar; deep_crustal_MASH [5,15) kbar; lithospheric_mantle [15,30) kbar;
deeper_mantle [30,100) kbar). Panels show test-set RMSE before vs after
correction per regime plus the pooled "ALL" row. Result: P_kbar
corrections improve ALL RMSE (MLP 3.82 to 2.58 kbar; ElasticNet 5.59
to 3.05 kbar). T_C correction is a neutral-to-negative wash on
ElasticNet/raw (pooled 77.06 to 77.17; shallow_crustal HURT 35.33 to
54.01), consistent with the earlier finding that train-OOF T bias does not
transfer to the held-out test set. Produced by
scripts/v10_phase_g_nb07_bias.py. Supports T09 (bias correction
effectiveness). Source CSVs: results/opx_liq_bias_correction.csv,
results/opx_liq_bias_correction_params.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig28_bias_correction_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig29_twopx_benchmark

Fig. 29. the tuned ML baseline two-pyroxene (twopx) benchmark, consolidating the per-cell
(single seed), 20-seed multiseed, and stacked-ensemble results on the
twopx Citation-grouped held-out test split. Bars show single-seed test
RMSE for the best base model (blue, ElasticNet/raw for T_C; XGB/alr
for P_kbar) and best stacked ensemble (orange, greedy/pwlr for both
targets). Whiskers on the base-model bars show the 20-seed RMSE
spread (min to max) from the final phase multiseed refit. The Putirka
2008 two-pyroxene equations (eq36/37/38/39) are DEFERRED because the
call requires paired opx-cpx Thermobar invocation that is not wired
into the tuned ML baseline; the deferred row is retained in the CSV with NaN RMSE so
downstream consumers (NB09 manuscript, NBF figures) flag the gap
explicitly. Fixes the earlier empty nb10_two_pyroxene_benchmark.csv
regression. Produced by scripts/v10_phase_g_nb10_twopx_benchmark.py.
Source CSVs: results/v10_twopx_benchmark_final.csv and
tables/S8_9_twopx_benchmark.{md,csv}.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig29_twopx_benchmark.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig32_bias_correction_form_comparison

Fig. 32. Test-set RMSE at regime=ALL comparing pre-correction with Form A and Form B post-correction for the 8 the final phase cells (20-seed mean). Shipped form (if any) is drawn with a bold border. Panel subtitle reports the final phase shipping decision and the per-seed vote split across 20 model seeds. Source CSVs: results/bias_correction_{summary,per_seed,shipped}.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig32_bias_correction_form_comparison.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig33_bias_correction_per_seed_stability

Fig. 33. Per-seed $\Delta$RMSE (pre minus post, positive = improvement) at regime=ALL for Form A (x-axis) vs Form B (y-axis), one point per model seed across the 20-seed protocol. Points are colored by the per-seed winner (blue = A, vermillion = B, gray = none). Light-green shading highlights the upper-right quadrant where both forms improve test RMSE; light-red shading highlights the lower-left quadrant where both forms degrade it. Panel annotation reports the per-seed vote count. Source CSV: results/bias_correction_per_seed.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig33_bias_correction_per_seed_stability.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig35_tabpfn_vs_opx_tb

Fig. 35. TabPFN v2 (Hollmann et al. 2025) versus the pre-registered tuned baseline and the best classical/ML external reference per (pipeline, track, target) combination. TabPFN error bars show 5-seed ensemble stability; the tuned ML baseline error bars show 20-seed model-fit variance. TabPFN receives raw oxide features only (no ALR/PWLR) and is fit with default hyperparameters (n_estimators=8 opx / 4 cpx, device=cpu). Sources: results/tabpfn_multiseed_summary.csv, results/tabpfn_head_to_head.csv, results/v10_{opx,cpx}_multiseed_summary.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig35_tabpfn_vs_opx_tb.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig44_tabpfn_bias_scoreboard_opx

Fig. 44. TabPFN v2 bias-correction scoreboard on the 4 opx combinations. Panel A: mean test RMSE (over 5 seeds) pre-correction (gray) and after applying Form A (blue, regime-piecewise linear) or Form B (orange, piecewise sigmoid) fit on 5-seed x 10-fold OOF residuals. Annotations show winner verdict under the conservative ship-if-better rule at canonical seed 42 (overall delta > 1e-6 and no regime degradation). Form A ships on opx_only/T_C and opx_only/P_kbar with large pre-post gaps (TabPFN materially over-predicts deeper mantle residual bias uncorrected). Form B ships nothing on opx TabPFN, consistent with the 0/8 tuned-family pattern. Panel B: per-seed ship stability over seeds 42-46; cells color winner (gray = none, blue = A, orange = B) annotated with the verdict. Shipping is consistent at seed 42, 43, 45 on opx_only/T_C and at all 5 seeds on opx_only/P_kbar. Sources: results/tabpfn_bias_correction_perseed.csv, results/tabpfn_bias_correction_summary.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig44_tabpfn_bias_scoreboard_opx.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig_aug01_ship_verdict_comparison

Ship-if-better verdict under the Agreda-Lopez (2024) 15x augmentation protocol versus the non-augmented baseline, across the four opx combinations (track x target). Bars show Form A / Form B / none ship counts across 20 seeds per condition.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig_aug01_ship_verdict_comparison.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig_aug02_aggregate_rmse_delta

Aggregate test RMSE under the non-augmented baseline versus the 15x augmented condition per opx combination. Error bars are 20-seed standard deviation. Putirka classical reference is overlaid where available.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig_aug02_aggregate_rmse_delta.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig_aug03_residual_structure_per_regime

OOF residual distributions by pressure regime for opx-only P_kbar, non-augmented (top row) versus 15x augmented (bottom row). Violin plots show Form A and Form B fit substrate under each protocol.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig_aug03_residual_structure_per_regime.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig_aug04_form_b_breakpoint_stability

Form B breakpoint positions (alpha_L, alpha_R) per seed for opx-only P_kbar under 15x augmentation. Tight clustering indicates stable Form B fits; scatter across the quantile grid indicates the fit drifts with data draws and explains Form B ship failures.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig_aug04_form_b_breakpoint_stability.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

## 3. Data tables

Every number above loads from a CSV under `results/`. Nothing is
hard-coded. If a file is missing the cell raises `FileNotFoundError`
rather than silently filling in a value.

### 3.1 Headline table

The headline numbers for each of the 4 opx cells, at the "ALL
pressure regimes" level. Columns: uncorrected tuned-ML RMSE,
bias-corrected tuned-ML RMSE, the best Putirka (2008)
thermobarometer for that cell, TabPFN pre- and post-correction
RMSE, and which of those five candidates wins.
Source: `results/preregistered_scorecard_postcorrection.csv`.

In [ ]:
sc = load_csv('results/preregistered_scorecard_postcorrection.csv')
hl = sc[(sc['track'].isin(['opx_liq','opx_only'])) & (sc['regime']=='ALL')].copy()
display_cols = ['track','target','v10_pre_rmse','v10_post_rmse','best_external_rmse',
                'tabpfn_rmse','tabpfn_post_rmse','tabpfn_correction_form','winner']
hl = hl[display_cols].rename(columns={
    'v10_pre_rmse': 'tuned pre',
    'v10_post_rmse': 'tuned post',
    'best_external_rmse': 'Putirka best',
    'tabpfn_rmse': 'TabPFN pre',
    'tabpfn_post_rmse': 'TabPFN post',
    'tabpfn_correction_form': 'TabPFN form',
})
display(hl)

### 3.2 Eight-cell winner table

Which model shipped, with or without bias correction, in each of
the 8 opx rows (4 tuned-family rows + 4 TabPFN rows).
Source: `results/bias_correction_shipped.csv`.

In [ ]:
shipped = load_csv('results/bias_correction_shipped.csv')
opx_rows = shipped[shipped['pipeline']=='opx'].copy()
display_cols = ['track','target','model','winner','ship_a','ship_b','n_seeds_done']
display(opx_rows[display_cols])

### 3.3 TabPFN bias-correction scoreboard

TabPFN's own pre/post correction numbers, averaged over 5 OOF
seeds. This is the source for the TabPFN columns in Section 3.1.
Source: `results/tabpfn_bias_correction_summary.csv`.

In [ ]:
tf_sum = load_csv('results/tabpfn_bias_correction_summary.csv')
display(tf_sum)

Per-seed detail (5 seeds x 4 combos = 20 rows). Useful for
checking stability.
Source: `results/tabpfn_bias_correction_perseed.csv`.

In [ ]:
tf_per = load_csv('results/tabpfn_bias_correction_perseed.csv')
display(tf_per[['track','target','seed','pre_rmse_all','post_rmse_a','post_rmse_b','ship_a','ship_b','winner']])

### 3.4 opx-only P_kbar per-regime breakdown

For the pressure cell where ML does best (opx-only P), here is
the regime-by-regime picture across all 5 candidates: tuned pre,
tuned post, Putirka eq. 29c, TabPFN pre, TabPFN post.

In [ ]:
pr = sc[(sc['track']=='opx_only') & (sc['target']=='P_kbar')].copy()
show = ['regime','n','v10_pre_rmse','v10_post_rmse','best_external_rmse',
        'tabpfn_rmse','tabpfn_post_rmse','tabpfn_correction_form','winner']
display(pr[show])

### 3.5 TabPFN head-to-head (aggregate RMSE, 20-seed baseline)

TabPFN vs the best tuned family per cell, at the aggregate ALL
level, using the full 20-seed multiseed protocol so the stability
estimates are on an equal footing with the tuned families.
Source: `results/tabpfn_head_to_head.csv`.

In [ ]:
h2h = load_csv('results/tabpfn_head_to_head.csv')
display(h2h)

## 4. Methods summary (plain English)

- **What we are predicting.** For each pyroxene sample we are
  trying to predict either (a) the temperature at which it formed
  or (b) the pressure at which it formed. Two input "tracks": the
  full crystal + liquid pair ("opx-liq") or the crystal alone
  ("opx-only"). Four target cells in total.
- **Where the training data come from.** A curated subset of the
  LEPR experimental petrology database (our snapshot is dated
  2025-07-21). Every sample is from a published experiment at
  known P and T. We group by citation when splitting into train
  and test, so that memorizing a single lab's style is not
  rewarded.
- **Which models we tune.** 8 standard ML families (random
  forest, extremely-randomized trees, XGBoost, gradient boosting,
  CatBoost, LightGBM, elastic-net linear, multi-layer
  perceptron). Each one is tuned with Optuna (50 trials) on a
  citation-grouped 5-fold cross-validation objective. We also
  run a 9th foundation-model baseline: TabPFN v2 (Hollmann et al.
  2025), which requires no tuning.
- **Pressure regimes.** Before we fit any correction, we
  registered four geologic pressure bins: shallow-crustal
  (<5 kbar), deep-crustal / MASH (5-15 kbar),
  lithospheric-mantle (15-30 kbar), deeper-mantle (30+ kbar).
  "ALL" is the full test set combined.
- **Bias correction.** Two flavors. *Form A* fits a simple linear
  regression of residual on predicted value, separately in each
  of the four regimes. *Form B* fits one smooth piecewise-
  sigmoid curve across the whole predicted-value axis. Only one
  flavor can ship per cell.
- **Ship-if-better rule (as of Amendment 1, 2026-04-20).** A
  correction ships if overall RMSE improves AND no
  pre-registered regime with enough samples (N >= 20) gets
  worse. Regimes with fewer than 20 samples are noted but are
  not allowed to veto a real improvement elsewhere. This
  replaces the original strict rule that any regime with any
  degradation would block shipping; the tiered rule is documented
  in `docs/preregistration/AMENDMENT_1_acceptance_rule.md`.
- **How we measure stability.** Each experiment is repeated at
  20 random seeds (42-61). The bars and numbers you see include
  bootstrap 95% confidence intervals so you can see whether a
  difference is signal or noise.

## 5. Limitations

1. **Temperature corrections mostly fail to ship.** In our
   strict evaluation, the bias correction refuses to ship for 3
   of 4 temperature cells. Temperature residuals do not have a
   strong regime structure to remove, so per-regime correction
   cancels out. We report this as a null result rather than
   tuning until something ships.
2. **opx-only T is the weakest cell.** For the opx-only
   thermometer, TabPFN improves the aggregate RMSE but the gain
   is marginal; we would not recommend that particular cell for
   deployment today.
3. **Natural-sample cross-check carries a small T bias.** On
   LEPR paired pyroxenes (n=327) the opx-only ML prediction
   runs about +80 C hotter than Putirka's two-pyroxene method
   and Jorgenson's cpx-only method. The pressure agreement is
   within our conformal error bar.
4. **No GEOROC refresh since 2026-04-09.** We did not pull a
   new natural-sample export for this round; the natural
   comparison uses the on-disk export from that date.
5. **SHAP on TabPFN is not available.** TabPFN is an in-context
   foundation model with no gradient surface exposed. For the
   two cells where TabPFN is the scorecard winner (opx_only T
   and opx_only P), Core_12 shows the tuned-family runner-up's
   SHAP for explainability and flags the swap in the subtitle.

## 6. Caveats and reconstruction provenance

Short file of "things a reviewer should know before acting on the
numbers." Loaded from `CAVEATS.md` alongside this notebook.

In [ ]:
cav = Path('./CAVEATS.md')
if cav.exists():
    display(Markdown(cav.read_text(encoding='utf-8')))
else:
    display(Markdown('(caveats file missing)'))

## 7. Provenance

Git state + source CSV sizes at build time. Loaded from
`PROVENANCE.md` alongside this notebook.

In [ ]:
prov = Path('./PROVENANCE.md')
if prov.exists():
    display(Markdown(prov.read_text(encoding='utf-8')))
else:
    display(Markdown('(provenance file missing)'))

## 8. Pre-registration (verbatim)

The rules of the game, frozen before we fit any corrections.
Reproduced verbatim from `docs/preregistration/`.

In [ ]:
# Prefer the local preregistration/ copy bundled with the package;
# fall back to the project docs/ source if running uninstalled.
for name in ('p_regime_preregistration.md', 'nb03_test_protocol.md'):
    local = Path('./preregistration') / name
    src = ROOT / 'docs' / 'preregistration' / name
    p = local if local.exists() else src
    if not p.exists():
        raise FileNotFoundError(f'preregistration missing: {name}')
    display(Markdown(f'### `{name}`'))
    display(Markdown(p.read_text(encoding='utf-8')))